# Exploration de l'API Frankfurter (taux de change BCE)

**Entreprise :** Bresse Mécanique SAS, sous-traitant industriel de 120 salariés à Bourg-en-Bresse.
Elle achète en dollars (USD), en yuans (CNY) et en francs suisses (CHF), et vend en livres sterling (GBP).

**Besoin métier :** le budget est construit avec des taux de change fixes (taux budgétaires).
En cours d'année, les taux réels varient. Le contrôle de gestion veut mesurer l'écart entre
taux réels et taux budgétaires, et son impact en euros sur les achats.

**Objectif de ce notebook :** découvrir pas à pas les données de l'API avant de les automatiser :
- comprendre la réponse de l'API ;
- la transformer en tableau ;
- tester les erreurs ;
- vérifier la qualité des données.

**Source :** API Frankfurter, taux de référence officiels de la Banque centrale européenne.
Elle est gratuite, ne demande pas de clé et n'impose pas de quota.

In [1]:
import requests
import pandas as pd

## 1. Bibliothèques utilisées

- **requests** : envoie les demandes à l'API par Internet et récupère sa réponse.
- **pandas** : transforme la réponse en tableau (DataFrame), comme une feuille Excel,
  pour pouvoir filtrer, compter et contrôler les données.

## 2. Premier appel à l'API : les taux du jour

On interroge l'API pour obtenir les **derniers taux publiés** par la BCE pour nos 4 devises.

- **Adresse appelée (endpoint)** : `https://api.frankfurter.dev/v2/providers/ecb/rates`
  → la route `providers/ecb` garantit des taux officiels de la BCE.
- **Paramètre** `quotes` : la liste des devises voulues, pour ne recevoir que les données utiles.
- **timeout=30** : si l'API ne répond pas en 30 secondes, on abandonne au lieu d'attendre indéfiniment.

URL = "https://api.frankfurter.dev/v2/providers/ecb/rates"
params = {"quotes": "USD,CNY,CHF,GBP"}

reponse = requests.get(URL, params=params, timeout=30)

print("URL appelée :", reponse.url)
print("Code HTTP   :", reponse.status_code)

reponse.json()

In [2]:
URL = "https://api.frankfurter.dev/v2/providers/ecb/rates"
params = {"quotes": "USD,CNY,CHF,GBP"}

reponse = requests.get(URL, params=params, timeout=30)

print("URL appelée :", reponse.url)
print("Code HTTP   :", reponse.status_code)

reponse.json()

URL appelée : https://api.frankfurter.dev/v2/providers/ecb/rates?quotes=USD%2CCNY%2CCHF%2CGBP
Code HTTP   : 200


[{'date': '2026-10-13', 'base': 'EUR', 'quote': 'CHF', 'rate': 0.9312},
 {'date': '2026-10-13', 'base': 'EUR', 'quote': 'CNY', 'rate': 7.489},
 {'date': '2026-10-13', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.8446},
 {'date': '2026-10-13', 'base': 'EUR', 'quote': 'USD', 'rate': 1.1188}]

**Lecture de la réponse**

- Code **200** : la demande a réussi.
- L'API renvoie une **liste** (entre crochets `[ ]`), avec **un élément par devise** (entre accolades `{ }`).
- Chaque élément contient 4 champs :
  - `date` : jour de publication du taux par la BCE ;
  - `base` : devise de départ, toujours l'euro (EUR) ;
  - `quote` : devise d'arrivée (USD, CNY, CHF, GBP) ;
  - `rate` : le taux, c'est-à-dire **combien d'unités de la devise pour 1 €**.
    Exemple : rate = 1,08 pour USD signifie 1 € = 1,08 $.

**Intérêt pour le contrôle de gestion** : pour convertir un achat en euros, on divise le montant
en devise par le taux. Une facture de 10 000 $ au taux du 12/10/2026 (1,1181) coûte 10 000 / 1,1181 ≈ 8 944 €.
Si le taux baisse, l'achat coûte plus cher en euros.

## 3. Récupérer l'historique depuis janvier 2025

Pour analyser l'évolution des taux et la comparer au budget, il faut un **historique**.
On ajoute le paramètre `from` (date de début) : l'API renvoie alors tous les taux
publiés depuis cette date, pour nos 4 devises.

In [3]:
params_histo = {"from": "2025-01-01", "quotes": "USD,CNY,CHF,GBP"}

reponse_histo = requests.get(URL, params=params_histo, timeout=30)
print("Code HTTP :", reponse_histo.status_code)

donnees = reponse_histo.json()
print("Nombre de lignes reçues :", len(donnees))

donnees[:3]

Code HTTP : 200
Nombre de lignes reçues : 1824


[{'date': '2024-12-31', 'base': 'EUR', 'quote': 'CHF', 'rate': 0.9412},
 {'date': '2024-12-31', 'base': 'EUR', 'quote': 'CNY', 'rate': 7.5833},
 {'date': '2024-12-31', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.82918}]

**Constat :** les premières lignes sont datées du **31/12/2024**, alors qu'on a demandé
à partir du 01/01/2025.
Le 1er janvier est férié, la BCE ne publie pas de taux ce jour-là : l'API renvoie donc
le dernier taux disponible avant la date demandée.
→ Ces lignes devront être **retirées lors du nettoyage**, car elles sont hors de la période d'analyse.

In [4]:
df = pd.DataFrame(donnees)

print("Dimensions (lignes, colonnes) :", df.shape)
print()
print(df.dtypes)

df.head()

Dimensions (lignes, colonnes) : (1824, 4)

date      object
base      object
quote     object
rate     float64
dtype: object


,date,base,quote,rate
0,2024-12-31,EUR,CHF,0.94120
1,2024-12-31,EUR,CNY,7.58330
2,2024-12-31,EUR,GBP,0.82918
3,2024-12-31,EUR,USD,1.03890
4,2025-01-02,EUR,CHF,0.93710


**Lecture du tableau**

- 4 colonnes : `date`, `base`, `quote`, `rate`.
- Types détectés par pandas :
  - `rate` est en **float64** (nombre décimal) → correct, on peut calculer avec ;
  - `date` est en **object** (du texte) → **à convertir en vraie date**,
    sinon impossible de trier, filtrer par période ou regrouper par mois correctement ;
  - `base` et `quote` sont en object (texte) → correct.

C'est la première transformation à prévoir dans le nettoyage : **le traitement des type**

## 4. Exploration de la qualité des données

Avant de nettoyer et d'automatiser, on vérifie l'état des données reçues.
Chaque contrôle ci-dessous sert à décider d'une règle de nettoyage.

### 4.1 Recherche des valeurs manquantes

On compte les cases vides dans chaque colonne.
Une ligne sans date ou sans taux est inutilisable pour l'analyse.

In [5]:
# Nombre de valeurs manquantes par colonne
df.isna().sum()

date     0
base     0
quote    0
rate     0
dtype: int64

### 4.2 Recherche des doublons

On vérifie qu'il n'existe qu'une seule ligne par **date et par devise** (clé métier).
Un doublon fausserait les moyennes et les impacts calculés dans le dashboard.

In [6]:
# Nombre de lignes ayant la même date et la même devise qu'une ligne précédente
df.duplicated(subset=["date", "quote"]).sum()

np.int64(0)

### 4.3 Contrôle des devises

On vérifie que les 4 devises sont présentes avec le même nombre de lignes.
Un nombre inférieur pour une devise signifierait des jours manquants.

In [7]:
# Nombre de lignes par devise
df["quote"].value_counts()

quote
CHF    456
CNY    456
GBP    456
USD    456
Name: count, dtype: int64

### 4.4 Contrôle des dates

On convertit temporairement la colonne date (actuellement du texte) en vraies dates, puis on vérifie :
- la période couverte ;
- la présence de dates antérieures au 01/01/2025 ;
- l'absence de cotation le week-end.

In [8]:
# Conversion du texte en dates (copie temporaire, df n'est pas modifié)
dates = pd.to_datetime(df["date"])

# Période couverte
print("Première date :", dates.min().date())
print("Dernière date :", dates.max().date())

# Lignes hors période (avant le 1er janvier 2025)
print("Lignes avant le 01/01/2025 :", (dates < "2025-01-01").sum())

# Lignes datées d'un samedi (5) ou d'un dimanche (6)
print("Lignes le week-end :", (dates.dt.dayofweek >= 5).sum())

Première date : 2024-12-31
Dernière date : 2026-10-13
Lignes avant le 01/01/2025 : 4
Lignes le week-end : 0


### 4.5 Bilan de l'exploration

| Contrôle | Résultat | Décision pour le nettoyage |
|---|---|---|
| Valeurs manquantes | 0 | Règle conservée par sécurité : suppression des lignes sans date ou sans taux |
| Doublons date + devise | 0 | Règle conservée par sécurité : une seule ligne par date et devise |
| Lignes par devise | ... pour chacune | Les 4 devises sont complètes |
| Dates avant 2025 | 4 | **À supprimer** (taux du 31/12/2024, hors période) |
| Cotations le week-end | 0 | Normal : la BCE ne publie pas le week-end, ces jours ne sont pas comblés |
| Type de la colonne date | texte (object) | **À convertir** en date |

**Conclusion :** la donnée BCE est très propre. Les règles de nettoyage sont tout de même
conservées, car l'automatisation tournera chaque jour sans surveillance :
une anomalie future sera détectée avant d'atteindre BigQuery.

## 5. Nettoyage des données

On applique les décisions issues de l'exploration.
On travaille sur une **copie** du tableau (`df_propre`) pour garder les données brutes intactes
et pouvoir comparer avant/après.
Ces étapes sont les mêmes que celles codées dans `src/transform.py` (T1 à T7).

In [9]:
# Copie du tableau brut : df reste intact
df_propre = df.copy()

print("Lignes avant nettoyage :", len(df_propre))

Lignes avant nettoyage : 1824


### 5.1 Renommage des colonnes (T1)

On renomme les colonnes en français, avec des noms explicites.
Ils seront plus lisibles dans BigQuery et dans le dashboard.

In [10]:
# Ancien nom : nouveau nom
df_propre = df_propre.rename(columns={
    "date": "date_taux",
    "base": "devise_base",
    "quote": "devise",
    "rate": "taux"
})

df_propre.columns

Index(['date_taux', 'devise_base', 'devise', 'taux'], dtype='object')

### 5.2 Conversion des types (T2)

- `date_taux` : texte → **date**, pour pouvoir trier, filtrer et regrouper par période ;
- `taux` : déjà en nombre décimal aujourd'hui. La conversion est conservée **par sécurité** :
  si l'API renvoyait un jour un taux sous forme de texte, il serait automatiquement converti en nombre ;
- devises : texte en **majuscules sans espaces**, pour éviter que "usd" et "USD" soient vus comme deux devises.

`errors="coerce"` : une valeur illisible devient vide au lieu de faire planter le programme
(elle est ensuite supprimée à l'étape 5.4, avec les valeurs manquantes).

In [11]:
# Texte -> date (sans l'heure)
df_propre["date_taux"] = pd.to_datetime(df_propre["date_taux"], errors="coerce").dt.date

# Texte -> nombre décimal
df_propre["taux"] = pd.to_numeric(df_propre["taux"], errors="coerce")

# Devises : suppression des espaces + majuscules
df_propre["devise"] = df_propre["devise"].astype(str).str.strip().str.upper()
df_propre["devise_base"] = df_propre["devise_base"].astype(str).str.strip().str.upper()

df_propre.dtypes

date_taux       object
devise_base     object
devise          object
taux           float64
dtype: object

### 5.3 Filtre des devises utiles (T3)

On ne garde que les 4 devises du périmètre de Bresse Mécanique.
Sécurité au cas où l'API renverrait d'autres devises.

In [12]:
DEVISES = ["USD", "CNY", "CHF", "GBP"]

avant = len(df_propre)
df_propre = df_propre[df_propre["devise"].isin(DEVISES)]

print("Lignes retirées (devises hors périmètre) :", avant - len(df_propre))

Lignes retirées (devises hors périmètre) : 0


### 5.4 Suppression des valeurs manquantes (T5)

Une ligne sans date ou sans taux est inexploitable : on la supprime.

In [13]:
avant = len(df_propre)
df_propre = df_propre.dropna(subset=["date_taux", "taux"])

print("Lignes retirées (date ou taux manquant) :", avant - len(df_propre))

Lignes retirées (date ou taux manquant) : 0


### 5.5 Suppression des dates hors période (T4)

On retire les lignes antérieures au 01/01/2025 (les taux du 31/12/2024 renvoyés par l'API).

In [14]:
DATE_DEBUT = pd.Timestamp("2025-01-01").date()

avant = len(df_propre)
df_propre = df_propre[df_propre["date_taux"] >= DATE_DEBUT]

print("Lignes retirées (avant le 01/01/2025) :", avant - len(df_propre))

Lignes retirées (avant le 01/01/2025) : 4


### 5.6 Suppression des doublons (T6)

On garde une seule ligne par couple date + devise.
`keep="last"` : en cas de doublon, on conserve la valeur la plus récente reçue.

In [15]:
avant = len(df_propre)
df_propre = df_propre.drop_duplicates(subset=["date_taux", "devise"], keep="last")

print("Doublons retirés :", avant - len(df_propre))

Doublons retirés : 0


### 5.7 Ajout des colonnes de traçabilité (T7)

- `source` : d'où vient la donnée ;
- `date_collecte` : quand elle a été récupérée (heure UTC).

Utile pour remonter à l'origine d'un chiffre et pour prouver le fonctionnement de l'automatisation.

In [16]:
df_propre["source"] = "BCE via API Frankfurter"
df_propre["date_collecte"] = pd.Timestamp.now(tz="UTC").floor("s")

# Tri par date puis devise, et renumérotation des lignes
df_propre = df_propre.sort_values(["date_taux", "devise"]).reset_index(drop=True)

df_propre.head()

,date_taux,devise_base,devise,taux,source,date_collecte
0,2025-01-02,EUR,CHF,0.93710,BCE via API Frankfurter,2026-10-13 18:23:36+00:00
1,2025-01-02,EUR,CNY,7.53380,BCE via API Frankfurter,2026-10-13 18:23:36+00:00
2,2025-01-02,EUR,GBP,0.83118,BCE via API Frankfurter,2026-10-13 18:23:36+00:00
3,2025-01-02,EUR,USD,1.03210,BCE via API Frankfurter,2026-10-13 18:23:36+00:00
4,2025-01-03,EUR,CHF,0.93620,BCE via API Frankfurter,2026-10-13 18:23:36+00:00


### 5.8 Résultat du nettoyage

In [17]:
print("Lignes avant nettoyage :", len(df))
print("Lignes après nettoyage :", len(df_propre))
print("Période :", df_propre["date_taux"].min(), "→", df_propre["date_taux"].max())

Lignes avant nettoyage : 1824
Lignes après nettoyage : 1820
Période : 2025-01-02 → 2026-10-13


**Bilan :** 1820 lignes brutes → 1816 lignes propres.
Seules les 4 lignes hors période ont été retirées ; aucune valeur manquante ni aucun doublon.
Le tableau est prêt à être chargé dans BigQuery.

**Remarque :** toutes les lignes ont la même `date_collecte`, car elles ont été récupérées en un seul appel
(chargement initial de l'historique). Lors des exécutions automatiques quotidiennes, chaque nouvelle ligne
portera sa propre date de collecte. L'heure est en UTC (France = UTC+2 en été).
Le taux BCE étant publié une seule fois par jour, `date_taux` ne contient pas d'heure.

## 6. Test des erreurs de l'API

On provoque volontairement des erreurs pour vérifier deux choses :
- comment l'API réagit (code HTTP et message renvoyés) ;
- que notre code les détecte **sans planter**.

Ces cas sont ceux gérés dans `src/extract.py`.

### 6.1 Création d'une fonction de test

Plutôt que de répéter le même code pour chaque test, on crée une **fonction** réutilisable.
Elle appelle l'API, affiche le code HTTP et le message d'erreur éventuel.

- `try / except` : si une erreur réseau survient, on l'affiche au lieu d'arrêter le notebook.
- Si le code n'est pas 200, on lit le message d'explication renvoyé par l'API.

In [18]:
def tester_appel(description, url, params, timeout=30):
    print("Test :", description)
    try:
        # Envoi de la demande à l'API
        rep = requests.get(url, params=params, timeout=timeout)

        if rep.status_code == 200:
            print("  → Succès (200)")
        else:
            # L'API renvoie un message d'explication au format JSON
            print("  → Erreur HTTP", rep.status_code, ":", rep.json().get("message"))

    except requests.exceptions.Timeout:
        print("  → Erreur : l'API n'a pas répondu à temps (timeout)")
    except requests.exceptions.ConnectionError:
        print("  → Erreur : impossible de joindre l'API (connexion)")
    print()

### 6.2 Appel correct (référence)

On vérifie d'abord qu'un appel normal fonctionne, pour pouvoir comparer avec les erreurs.

In [19]:
tester_appel("appel normal", URL, {"quotes": "USD,CNY,CHF,GBP"})

Test : appel normal
  → Succès (200)



### 6.3 Devise inconnue

On demande une devise qui n'existe pas (`ABC`).

In [20]:
tester_appel("devise inconnue", URL, {"quotes": "USD,ABC"})

Test : devise inconnue
  → Erreur HTTP 422 : invalid currency: ABC



### 6.4 Date invalide

On demande une date impossible (13e mois, 45e jour).

In [21]:
tester_appel("date invalide", URL, {"from": "2025-13-45", "quotes": "USD"})

Test : date invalide
  → Erreur HTTP 422 : invalid date



### 6.5 Adresse inexistante

On appelle une route qui n'existe pas dans l'API.

In [22]:
tester_appel("adresse inexistante", "https://api.frankfurter.dev/v2/route_inexistante", {})

Test : adresse inexistante
  → Erreur HTTP 404 : not found



### 6.6 Délai de réponse dépassé (timeout)

On simule une API trop lente en imposant un délai d'attente minuscule (0,001 seconde).

In [23]:
tester_appel("délai dépassé", URL, {"quotes": "USD"}, timeout=0.001)

Test : délai dépassé
  → Erreur : l'API n'a pas répondu à temps (timeout)



### 6.7 Bilan des tests

| Test | Réponse obtenue | Signification |
|---|---|---|
| Appel normal | 200 | Succès |
| Devise inconnue | ... | Demande comprise mais impossible à traiter |
| Date invalide | ... | Paramètre mal formé |
| Adresse inexistante | ... | Ressource introuvable |
| Délai dépassé | timeout | L'API n'a pas répondu à temps |

**Conclusion :** dans chaque cas, l'erreur est détectée et expliquée **sans faire planter le programme**.
Dans `extract.py`, ces erreurs sont en plus **écrites dans le fichier de log** et le script se termine
en échec (code 1), ce qui permettra à l'automatisation de signaler un lancement raté.

## 7. Passage en production : du notebook aux scripts

Le code validé dans ce notebook est repris dans des scripts Python,
pour pouvoir être exécuté automatiquement chaque jour, sans intervention.

| Sections du notebook | Script | Ajouts pour l'automatisation |
|---|---|---|
| 2, 3 et 6 (appel API, historique, erreurs) | `src/extract.py` | fonctions, logs, code d'échec, sauvegarde du JSON brut |
| 4 et 5 (exploration, nettoyage T1 à T7) | `src/transform.py` | contrôles qualité automatiques (CQ1 à CQ5), sauvegarde du CSV propre |
| — | `src/load.py` | chargement dans BigQuery sans doublons |

### 7.1 Vérification : le script donne le même résultat que le notebook

On compare le tableau nettoyé dans ce notebook avec le fichier produit par `transform.py`.

In [24]:
# Lecture du fichier produit par transform.py
df_script = pd.read_csv("../data/processed/taux_propres.csv")

print("Lignes dans le notebook :", len(df_propre))
print("Lignes produites par transform.py :", len(df_script))

# Comparaison des taux, à date et devise égales
comparaison = (
    df_propre[["date_taux", "devise", "taux"]].astype(str)
    .merge(df_script[["date_taux", "devise", "taux"]].astype(str),
           on=["date_taux", "devise"], suffixes=("_notebook", "_script"))
)
print("Taux identiques :", (comparaison["taux_notebook"] == comparaison["taux_script"]).all())

Lignes dans le notebook : 1820
Lignes produites par transform.py : 1816
Taux identiques : True


## 8. Chargement des données dans BigQuery

On envoie le tableau nettoyé (`df_propre`) dans BigQuery, l'entrepôt de données cloud de Google.
Étapes :
1. connexion à BigQuery avec le compte de service ;
2. création de la table et de sa structure ;
3. lecture des lignes déjà présentes ;
4. filtre anti-doublons ;
5. chargement et vérification.

Ce code sera ensuite repris dans `src/load.py` pour l'automatisation.

### 8.1 Connexion à BigQuery

- **google-cloud-bigquery** : bibliothèque officielle de Google pour dialoguer avec BigQuery en Python.
- La connexion utilise la **clé du compte de service** `pipeline-taux` (fichier JSON).
  Ce compte n'a que deux droits : écrire des données et lancer des requêtes dans BigQuery
  (principe du moindre privilège).
- Le fichier de clé est un secret : il est exclu de GitHub par le `.gitignore`.

In [25]:
from google.cloud import bigquery

# Paramètres du projet
PROJECT_ID = "bresse-fx"            # ID du projet Google Cloud
DATASET = "bresse_fx"               # ensemble de données (dataset)
TABLE_ID = f"{PROJECT_ID}.{DATASET}.taux_change"   # nom complet de la table

# Chemin de la clé : "../" car le notebook est dans le dossier notebooks/
CHEMIN_CLE = "../cle_service_bigquery.json"

# Connexion à BigQuery avec le compte de service
client = bigquery.Client.from_service_account_json(CHEMIN_CLE, project=PROJECT_ID)

# Test : liste des datasets du projet
print("Connecté au projet :", client.project)
print("Datasets disponibles :", [d.dataset_id for d in client.list_datasets()])

Connecté au projet : bresse-fx
Datasets disponibles : ['bresse_fx']


### 8.2 Création de la table et de sa structure (schéma)

Le **schéma** définit le nom et le **type** de chaque colonne dans BigQuery :
- `DATE` pour la date du taux, `FLOAT64` pour le taux (nombre décimal),
  `STRING` pour le texte, `TIMESTAMP` pour la date et l'heure de collecte ;
- `REQUIRED` : la colonne ne peut pas être vide.

Définir le schéma nous-mêmes (plutôt que de laisser BigQuery le deviner) garantit des types corrects
pour les calculs SQL et le dashboard.

`exists_ok=True` : si la table existe déjà, on ne la recrée pas (pas d'erreur, pas de perte de données).

In [26]:
SCHEMA = [
    bigquery.SchemaField("date_taux", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("devise_base", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("devise", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("taux", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("source", "STRING"),
    bigquery.SchemaField("date_collecte", "TIMESTAMP"),
]

# Création de la table (seulement si elle n'existe pas encore)
table = bigquery.Table(TABLE_ID, schema=SCHEMA)
client.create_table(table, exists_ok=True)

# Vérification
table = client.get_table(TABLE_ID)
print("Table :", table.full_table_id)
print("Nombre de lignes :", table.num_rows)
for colonne in table.schema:
    print(f"  - {colonne.name} : {colonne.field_type}")

Table : bresse-fx:bresse_fx.taux_change
Nombre de lignes : 1816
  - date_taux : DATE
  - devise_base : STRING
  - devise : STRING
  - taux : FLOAT
  - source : STRING
  - date_collecte : TIMESTAMP


### 8.3 Lecture des lignes déjà présentes dans BigQuery

Avant de charger, on demande à BigQuery quelles combinaisons **date + devise** il contient déjà.
C'est la base du mécanisme anti-doublons : l'automatisation tournera chaque jour
et ne doit jamais réinsérer une ligne existante.

In [27]:
# Requête SQL : liste des couples (date, devise) déjà présents
requete = f"SELECT DISTINCT date_taux, devise FROM `{TABLE_ID}`"

# Exécution et stockage dans un ensemble (set) pour une recherche rapide
cles_existantes = {(ligne.date_taux, ligne.devise) for ligne in client.query(requete).result()}

print("Lignes déjà présentes dans BigQuery :", len(cles_existantes))

Lignes déjà présentes dans BigQuery : 1816


In [28]:
# Requête SQL : liste des couples (date, devise) déjà présents
requete = f"SELECT DISTINCT date_taux, devise FROM `{TABLE_ID}`"

# Exécution et stockage dans un ensemble (set) pour une recherche rapide
cles_existantes = {(ligne.date_taux, ligne.devise) for ligne in client.query(requete).result()}

print("Lignes déjà présentes dans BigQuery :", len(cles_existantes))

Lignes déjà présentes dans BigQuery : 1816


### 8.4 Filtre anti-doublons

On ne garde que les lignes dont la clé (date + devise) **n'est pas encore** dans BigQuery.

**Pourquoi en Python et pas en SQL ?** Le mode gratuit de BigQuery (bac à sable) n'autorise pas
les requêtes qui modifient une table (`INSERT`, `UPDATE`, `MERGE`). On filtre donc les nouvelles
lignes en Python, puis on les ajoute par un « chargement par lot », qui lui est autorisé.

In [29]:
# Pour chaque ligne : True si sa clé (date, devise) existe déjà dans BigQuery
deja_presente = [
    (d, dev) in cles_existantes
    for d, dev in zip(df_propre["date_taux"], df_propre["devise"])
]

# On garde les lignes qui ne sont PAS déjà présentes
nouvelles = df_propre[[not x for x in deja_presente]]

print("Lignes nettoyées :", len(df_propre))
print("Déjà présentes (ignorées) :", sum(deja_presente))
print("Nouvelles lignes à charger :", len(nouvelles))

Lignes nettoyées : 1820
Déjà présentes (ignorées) : 1816
Nouvelles lignes à charger : 4


### 8.5 Chargement dans BigQuery

- `load_table_from_dataframe` : envoie le tableau pandas dans la table BigQuery ;
- `WRITE_APPEND` : les lignes sont **ajoutées** à la fin de la table, sans effacer l'existant ;
- `job.result()` : on attend la fin du chargement (une erreur s'affiche s'il échoue).

In [30]:
if nouvelles.empty:
    print("Aucune nouvelle ligne à charger")
else:
    config = bigquery.LoadJobConfig(schema=SCHEMA, write_disposition="WRITE_APPEND")
    job = client.load_table_from_dataframe(nouvelles, TABLE_ID, job_config=config)
    job.result()   # attente de la fin du chargement
    print("Lignes chargées :", len(nouvelles))

print("Total dans la table :", client.get_table(TABLE_ID).num_rows)

Lignes chargées : 4
Total dans la table : 1820


### 8.6 Vérification par une requête SQL

On interroge directement BigQuery pour contrôler le contenu chargé :
nombre de lignes, première et dernière date, par devise.

In [31]:
requete_verif = f"""
SELECT
  devise,
  COUNT(*)        AS nb_lignes,
  MIN(date_taux)  AS premiere_date,
  MAX(date_taux)  AS derniere_date,
  ROUND(AVG(taux), 4) AS taux_moyen
FROM `{TABLE_ID}`
GROUP BY devise
ORDER BY devise
"""

client.query(requete_verif).to_dataframe()

C:\Users\mouna\Documents\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,devise,nb_lignes,premiere_date,derniere_date,taux_moyen
0,CHF,455,2025-01-02,2026-10-13,0.9314
1,CNY,455,2025-01-02,2026-10-13,8.0234
2,GBP,455,2025-01-02,2026-10-13,0.8594
3,USD,455,2025-01-02,2026-10-13,1.1429


### 8.7 Bilan du chargement

- Table `bresse-fx.bresse_fx.taux_change` créée avec un schéma typé (DATE, STRING, FLOAT64, TIMESTAMP).
- **1 816 lignes chargées** : 4 devises × 454 jours de cotation, du 02/01/2025 au 12/10/2026.
- Anti-doublons : seules les lignes absentes de BigQuery (clé date + devise) sont chargées.

Ce code est repris dans `src/load.py` pour l'automatisation quotidienne.

## 9. Chargement de la table budget (référentiel interne)

Pour comparer les taux réels au budget, on ajoute une seconde table : `budget_devises`.
Elle contient les **données internes** de Bresse Mécanique :
- le **taux budgétaire** par devise et par année ;
- le **sens** du flux (achat ou vente) et sa catégorie ;
- le **montant mensuel** prévu dans chaque devise.

**Méthode de fixation du taux budget :** taux moyen observé en décembre de l'année précédente
(budget 2026 = moyenne de décembre 2025). Faute d'historique 2024, le budget 2025 est fixé
sur la moyenne de janvier 2025. Les montants mensuels sont des hypothèses de l'entreprise fictive.

### 9.1 Lecture du fichier budget

In [32]:
# Lecture du référentiel budget ("../" : le fichier est à la racine du projet)
df_budget = pd.read_csv("../referentiel/budget_devises.csv")

df_budget

,devise,annee,sens,categorie,taux_budget,montant_mensuel_devise
0,USD,2025,achat,Aluminium et acier,1.04,400000
1,USD,2026,achat,Aluminium et acier,1.17,400000
2,CNY,2025,achat,Composants electroniques,7.56,1500000
3,CNY,2026,achat,Composants electroniques,8.25,1500000
4,CHF,2025,achat,Outillage,0.94,60000
5,CHF,2026,achat,Outillage,0.93,60000
6,GBP,2025,vente,Client britannique,0.84,150000
7,GBP,2026,vente,Client britannique,0.88,150000


### 9.2 Chargement dans BigQuery

- Schéma défini explicitement : `annee` en nombre entier, `taux_budget` en décimal, etc.
- `WRITE_TRUNCATE` : la table est **entièrement remplacée** à chaque chargement.
  C'est adapté à un référentiel : si la direction révise le budget, on recharge le fichier
  et l'ancienne version disparaît, sans risque de doublons.

In [33]:
TABLE_BUDGET = f"{PROJECT_ID}.{DATASET}.budget_devises"

SCHEMA_BUDGET = [
    bigquery.SchemaField("devise", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("annee", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("sens", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("categorie", "STRING"),
    bigquery.SchemaField("taux_budget", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("montant_mensuel_devise", "FLOAT64", mode="REQUIRED"),
]

config = bigquery.LoadJobConfig(schema=SCHEMA_BUDGET, write_disposition="WRITE_TRUNCATE")
job = client.load_table_from_dataframe(df_budget, TABLE_BUDGET, job_config=config)
job.result()   # attente de la fin du chargement

print("Lignes dans budget_devises :", client.get_table(TABLE_BUDGET).num_rows)

Lignes dans budget_devises : 8
